# Model Evaluation and Probability Calibration — src Refactor

This notebook evaluates probability calibration on the validation set and then performs a **single final evaluation** on the untouched test set.

Reusable calibration, metric, and plot utilities are imported from `src/`.

## 1. Bootstrap project imports

In [ ]:
from pathlib import Path
import sys

# Make imports work whether VS Code starts the notebook from project root
# or from the notebooks/ directory.
cwd = Path.cwd().resolve()

if (cwd / "src").exists():
    PROJECT_DIR = cwd
elif (cwd.parent / "src").exists():
    PROJECT_DIR = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing src/. "
        "Open this notebook from the clinical-outcome-prediction project."
    )

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Project root:", PROJECT_DIR)
print("Python:", sys.executable)

## 2. Imports

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import (
    MODELS_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    PREDICTIONS_DIR,
    SELECTED_MODEL_PATH,
    SELECTED_MODEL_CONFIG_PATH,
)
from src.data import (
    load_modeling_splits,
    load_feature_config,
)
from src.models import (
    fit_sigmoid_calibrator,
    fit_isotonic_calibrator,
    compare_calibration_methods,
)
from src.evaluation import (
    classification_metrics,
    probability_metrics,
    plot_roc_curve,
    plot_precision_recall_curve,
    plot_calibration_curve,
    plot_confusion_matrix,
    plot_probability_distribution,
)

DEFAULT_THRESHOLD = 0.50

SIGMOID_MODEL_PATH = MODELS_DIR / "selected_model_sigmoid_calibrated.joblib"
ISOTONIC_MODEL_PATH = MODELS_DIR / "selected_model_isotonic_calibrated.joblib"
FINAL_CALIBRATED_MODEL_PATH = MODELS_DIR / "calibrated_selected_model.joblib"
CALIBRATION_CONFIG_PATH = MODELS_DIR / "calibration_config.json"

for directory in [
    MODELS_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    PREDICTIONS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

## 3. Load data and selected model

In [ ]:
train_df, validation_df, test_df = load_modeling_splits()
feature_config = load_feature_config()

with open(
    SELECTED_MODEL_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as file:
    selected_model_config = json.load(file)

selected_model = joblib.load(
    SELECTED_MODEL_PATH
)

target_column = feature_config["target_column"]
feature_columns = feature_config["retained_feature_columns"]
selected_model_name = selected_model_config["selected_model_name"]

X_validation = validation_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

y_validation = validation_df[target_column].copy()
y_test = test_df[target_column].copy()

print("Selected model:", selected_model_name)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)

## 4. Uncalibrated validation probability quality

In [ ]:
uncalibrated_validation_probability = (
    selected_model.predict_proba(
        X_validation
    )[:, 1]
)

pd.Series(
    probability_metrics(
        y_validation,
        uncalibrated_validation_probability,
    ),
    name="Uncalibrated validation",
)

## 5. Fit sigmoid and isotonic calibration on validation data

In [ ]:
sigmoid_calibrator = fit_sigmoid_calibrator(
    selected_model,
    X_validation,
    y_validation,
)

isotonic_calibrator = fit_isotonic_calibrator(
    selected_model,
    X_validation,
    y_validation,
)

sigmoid_validation_probability = (
    sigmoid_calibrator.predict_proba(
        X_validation
    )[:, 1]
)

isotonic_validation_probability = (
    isotonic_calibrator.predict_proba(
        X_validation
    )[:, 1]
)

validation_calibration_comparison = (
    compare_calibration_methods(
        selected_model,
        sigmoid_calibrator,
        isotonic_calibrator,
        X_validation,
        y_validation,
    )
)

validation_calibration_comparison

## 6. Validation calibration curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

plot_calibration_curve(
    y_validation,
    uncalibrated_validation_probability,
    name="Uncalibrated",
    n_bins=5,
    title="Validation Calibration Comparison",
    ax=ax,
)

plot_calibration_curve(
    y_validation,
    sigmoid_validation_probability,
    name="Sigmoid",
    n_bins=5,
    title="Validation Calibration Comparison",
    ax=ax,
)

plot_calibration_curve(
    y_validation,
    isotonic_validation_probability,
    name="Isotonic",
    n_bins=5,
    title="Validation Calibration Comparison",
    ax=ax,
)

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "validation_calibration_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 7. Select calibration method using validation Brier score

In [ ]:
best_row = (
    validation_calibration_comparison
    .sort_values(
        ["brier_score", "log_loss"]
    )
    .iloc[0]
)

selected_calibration_method = str(
    best_row["method"]
)

calibration_candidates = {
    "Uncalibrated": selected_model,
    "Sigmoid": sigmoid_calibrator,
    "Isotonic": isotonic_calibrator,
}

final_model = calibration_candidates[
    selected_calibration_method
]

print(
    "Selected validation calibration method:",
    selected_calibration_method,
)

print(
    "Important: final test performance will be reported "
    "without using the test set to choose this method."
)

## 8. Final untouched test evaluation

In [ ]:
uncalibrated_test_probability = (
    selected_model.predict_proba(
        X_test
    )[:, 1]
)

final_test_probability = (
    final_model.predict_proba(
        X_test
    )[:, 1]
)

uncalibrated_test_metrics = {
    "model": selected_model_name,
    "calibration_method": "Uncalibrated",
    **probability_metrics(
        y_test,
        uncalibrated_test_probability,
    ),
    **{
        key: value
        for key, value in classification_metrics(
            y_test,
            uncalibrated_test_probability,
            threshold=DEFAULT_THRESHOLD,
        ).items()
        if key not in {"auroc", "auprc"}
    },
}

final_test_metrics = {
    "model": selected_model_name,
    "calibration_method": selected_calibration_method,
    **probability_metrics(
        y_test,
        final_test_probability,
    ),
    **{
        key: value
        for key, value in classification_metrics(
            y_test,
            final_test_probability,
            threshold=DEFAULT_THRESHOLD,
        ).items()
        if key not in {"auroc", "auprc"}
    },
}

final_test_comparison = pd.DataFrame(
    [
        uncalibrated_test_metrics,
        final_test_metrics,
    ]
)

final_test_comparison

## 9. Final test ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

plot_roc_curve(
    y_test,
    uncalibrated_test_probability,
    name="Uncalibrated",
    title="Final Test ROC Curves",
    ax=ax,
)

if selected_calibration_method != "Uncalibrated":
    plot_roc_curve(
        y_test,
        final_test_probability,
        name=selected_calibration_method,
        title="Final Test ROC Curves",
        ax=ax,
    )

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "final_test_roc_curves.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 10. Final test precision-recall curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

plot_precision_recall_curve(
    y_test,
    uncalibrated_test_probability,
    name="Uncalibrated",
    title="Final Test Precision-Recall Curves",
    ax=ax,
)

if selected_calibration_method != "Uncalibrated":
    plot_precision_recall_curve(
        y_test,
        final_test_probability,
        name=selected_calibration_method,
        title="Final Test Precision-Recall Curves",
        ax=ax,
    )

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "final_test_pr_curves.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 11. Final calibration curve and confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

plot_calibration_curve(
    y_test,
    uncalibrated_test_probability,
    name="Uncalibrated",
    n_bins=4,
    title="Final Test Calibration Curves",
    ax=ax,
)

if selected_calibration_method != "Uncalibrated":
    plot_calibration_curve(
        y_test,
        final_test_probability,
        name=selected_calibration_method,
        n_bins=4,
        title="Final Test Calibration Curves",
        ax=ax,
    )

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "final_test_calibration_curves.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
plot_confusion_matrix(
    y_test,
    final_test_probability,
    threshold=DEFAULT_THRESHOLD,
    title=f"Final Test Confusion Matrix — {selected_calibration_method}",
    save_path=FIGURES_DIR / "final_test_confusion_matrix.png",
)

## 12. Save calibration artifacts and final test results

In [ ]:
joblib.dump(
    sigmoid_calibrator,
    SIGMOID_MODEL_PATH,
)

joblib.dump(
    isotonic_calibrator,
    ISOTONIC_MODEL_PATH,
)

joblib.dump(
    final_model,
    FINAL_CALIBRATED_MODEL_PATH,
)

validation_calibration_comparison.to_csv(
    TABLES_DIR / "validation_calibration_comparison.csv",
    index=False,
)

final_test_comparison.to_csv(
    TABLES_DIR / "final_test_metrics.csv",
    index=False,
)

final_test_predictions = test_df[
    ["subject_id", "hadm_id", "stay_id", target_column]
].copy()

final_test_predictions["uncalibrated_probability"] = (
    uncalibrated_test_probability
)
final_test_predictions["final_probability"] = (
    final_test_probability
)
final_test_predictions["final_prediction_0_50"] = (
    final_test_probability >= DEFAULT_THRESHOLD
).astype(int)

final_test_predictions.to_csv(
    PREDICTIONS_DIR / "final_test_predictions.csv",
    index=False,
)

calibration_config = {
    "base_model_name": selected_model_name,
    "selected_calibration_method": selected_calibration_method,
    "selection_dataset": "validation",
    "selection_metric": "Brier score",
    "test_set_used_for_calibration_selection": False,
}

with open(
    CALIBRATION_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        calibration_config,
        file,
        indent=2,
    )

print("Notebook 8 outputs saved.")

## Notebook 8 summary

Calibration was selected on validation data; test data was used only for final evaluation.

For your current demo cohort, if isotonic looks excellent on validation but degrades on test, document it as **calibration overfitting due to the tiny calibration cohort**.

Next: **Notebook 9 — SHAP interpretation of the uncalibrated Random Forest**.